# Live exercise: the ladder of ten, worked solutions

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## 1. Every column of the first 10 plays

In [ ]:
run("SELECT * FROM plays LIMIT 10")

## 2. Track name and minutes, for all plays

2,183 rows come back. The helper only shows the first ten.

In [ ]:
run("SELECT track_name, minutes_played FROM plays")

## 3. Only the plays longer than 9 minutes

32 rows.

In [ ]:
run("""
SELECT track_name, minutes_played
FROM plays
WHERE minutes_played > 9
ORDER BY minutes_played DESC
""")

## 4. Only the plays on the speaker, in 2025

Two conditions joined with `AND`. 174 rows.

In [ ]:
run("""
SELECT played_at, track_name, minutes_played
FROM plays
WHERE device = 'speaker'
  AND played_at BETWEEN '2025-01-01' AND '2025-12-31'
""")

## 5. The same, longest first

In [ ]:
run("""
SELECT played_at, track_name, minutes_played
FROM plays
WHERE device = 'speaker'
  AND played_at BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY minutes_played DESC
""")

## 6. How many plays were skipped?

In [ ]:
run("SELECT COUNT(*) AS skipped_plays FROM plays WHERE skipped = 1")

## 7. How many different devices?

Two ways to ask, and both are worth knowing. One gives you the number, the
other shows you what they are.

In [ ]:
run("SELECT COUNT(DISTINCT device) AS devices FROM plays")

In [ ]:
run("SELECT DISTINCT device FROM plays")

## 8. Plays per device, biggest first

In [ ]:
run("""
SELECT device, COUNT(*) AS plays
FROM plays
GROUP BY device
ORDER BY plays DESC
""")

## 9. Average minutes per device

Sorted by the average rather than the count, and the car comes out top.

That is a small surprise worth noticing: people skip less in the car,
probably because reaching for the phone is inconvenient. The dataset cannot
prove that, but it is the kind of question a result like this should prompt.

In [ ]:
run("""
SELECT device,
       COUNT(*)                      AS plays,
       ROUND(AVG(minutes_played), 2) AS avg_min
FROM plays
GROUP BY device
ORDER BY avg_min DESC
""")

## 10. Only the devices with more than 300 plays

`COUNT(*)` is an aggregate, so the filter has to be `HAVING` and not `WHERE`:
at `WHERE` time no counting has happened yet.

Try changing `HAVING` to `WHERE` to see the error. It is a clear one.

In [ ]:
run("""
SELECT device, COUNT(*) AS plays
FROM plays
GROUP BY device
HAVING COUNT(*) > 300
ORDER BY plays DESC
""")

In [ ]:
con.close()
print("done")